# Run the Where's Waldo pipeline on Kaggle

This notebook only *drives* the project: it runs the five project notebooks (`01` → `05`), unmodified, from
the uploaded package. See `kaggle/README.md` in the project for the full walkthrough.

**Before running** (right-hand sidebar → *Session options*):
1. **Accelerator: `GPU T4 x2`** — *not* P100 (Kaggle's PyTorch has no kernels for it).
2. **Internet: On** (needed for `pip install` and the YOLO weights download).
3. **Add Input** → your private dataset containing `wheres-waldo-project.zip`.

Then **Save Version → Save & Run All (Commit)** and close the tab; everything runs on Kaggle's servers.
The deliverables appear under **Output → `results/`**.


In [ ]:
import glob
import importlib.util
import shutil
import zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")

# Kaggle extracts uploaded zips into /kaggle/input/<dataset>/, but don't rely on the exact nesting.
hits = glob.glob(str(INPUT / "**" / "run_pipeline.py"), recursive=True)
runner_path, scratch = None, Path("/kaggle/working/_runner")
if hits:
    runner_path = Path(hits[0])
else:
    for z in glob.glob(str(INPUT / "**" / "*.zip"), recursive=True):
        with zipfile.ZipFile(z) as zf:
            if "kaggle/run_pipeline.py" in zf.namelist():
                zf.extract("kaggle/run_pipeline.py", scratch)
                runner_path = scratch / "kaggle" / "run_pipeline.py"
                break
assert runner_path is not None, (
    "Couldn't find the project package under /kaggle/input. Did you attach the dataset "
    "(Add Input -> your dataset)?"
)

spec = importlib.util.spec_from_file_location("run_pipeline", runner_path)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
shutil.rmtree(scratch, ignore_errors=True)

status = runner.run()


In [ ]:
import json

print(json.dumps(status["notebooks"], indent=1))
print("\nALL NOTEBOOKS OK" if status["ok"] else f"\nFAILED at {status.get('failed_at')} -- see the log above and results/notebooks/")
